# 09 — Hyperparameter Tuning
GridSearchCV on Logistic Regression, RandomizedSearchCV on Decision Tree.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, f1_score
from scipy.stats import randint
from src.preprocess import build_preprocessor
from src.config import RANDOM_STATE

In [ ]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()

## 1. GridSearchCV on Logistic Regression

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__solver': ['liblinear', 'lbfgs']
}

grid_lr = GridSearchCV(lr_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_lr.fit(X_train, y_train)

print(f"Best params: {grid_lr.best_params_}")
print(f"Best CV F1: {grid_lr.best_score_:.4f}")

y_pred_lr = grid_lr.predict(X_test)
print(f"\nTest F1 (class 1): {f1_score(y_test, y_pred_lr, pos_label=1):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=['No Default', 'Default']))

## 2. RandomizedSearchCV on Decision Tree
Randomized search is faster when the parameter space is large.
Instead of trying all combinations, it samples n_iter random ones.

In [ ]:
dt_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

param_dist = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf': randint(1, 10)
}

random_dt = RandomizedSearchCV(
    dt_pipeline, param_dist, n_iter=20, cv=5, scoring='f1',
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
random_dt.fit(X_train, y_train)

print(f"Best params: {random_dt.best_params_}")
print(f"Best CV F1: {random_dt.best_score_:.4f}")

y_pred_dt = random_dt.predict(X_test)
print(f"\nTest F1 (class 1): {f1_score(y_test, y_pred_dt, pos_label=1):.4f}")
print(classification_report(y_test, y_pred_dt, target_names=['No Default', 'Default']))

## 3. Grid vs Randomized — When to Use Each
| Method | When to Use |
|--------|------------|
| GridSearchCV | Small parameter space (< 20 combinations) |
| RandomizedSearchCV | Large space; want good-enough params fast |
| Both | Always use CV (never tune on test set!) |